# 04a — Exploración de metadata: V-Dem v16 (Full+Others)

**Objetivo del notebook:** entender la estructura del CSV de V-Dem v16 (dimensiones, columnas, tipos, valores) antes de decidir qué variables extraer para el TFM. Además extraer metadata del coodebook y tabularla.

El codebook se usa solo como referencia conceptual para interpretar el significado de cada variable una vez identificada.

**Salida principal:** una tabla de metadata (`metadata_vdem_v16.xlsx`) con una fila por columna del CSV, siguiendo el mismo patrón que `01_catalogo_indicadores.ipynb` (WDI).



## Bloque 1 — Setup y lectura de cabecera

**Objetivo:** cargar librerías y leer únicamente los nombres de columnas del CSV, sin cargar los datos en memoria.

**Justificación metodológica:** con un archivo de ~397 MB y potencialmente miles de columnas, el primer paso debe tener coste mínimo. `nrows=0` le pide a pandas que lea solo la cabecera, sin materializar ninguna fila de datos.

**Resultado esperado:** un `Index` con los nombres de columnas y el número total de columnas.


**Errores habituales:** `FileNotFoundError` si la ruta está mal escrita; error de encoding (poco probable en este archivo, pero si aparece, añade `encoding="utf-8"` explícito a la llamada).


In [1]:
import pandas as pd
import numpy as np
import re
import time
import plotly.express as px

# El cuaderno y la base tienen que estar en la misma carpeta para que esto funcione:
VDEM_PATH = "V-Dem-CY-Full+Others-v16.csv"

# Lista para acumular figuras, siguiendo el mismo patrón que en 02_disponibilidad_datos_wdi.ipynb
figuras_reporte = []

# Lectura de solo cabecera (coste de memoria mínimo)
columnas = pd.read_csv(VDEM_PATH, nrows=0).columns

print(f"Número total de columnas: {len(columnas)}")


Número total de columnas: 4618


## Bloque 2 — Conteo de filas

**Objetivo:** obtener el número de filas del CSV sin materializar el DataFrame completo.


**Resultado esperado:** un entero con el número de filas (sin contar la cabecera).

**Qué comprobar:** que el orden de magnitud tenga sentido. El V-Dem Full+Others cubre 202 unidades país/histórico entre 1789–2025, por lo que esperaríamos un rango aproximado de 30.000–45.000 filas (país-año), no millones.

**Errores habituales:** si el archivo tuviera saltos de línea dentro de campos de texto entrecomillados el conteo simple podría fallar (poco probable en este CSV, ya que casi todos los campos son numéricos). Si el resultado no cuadra con lo esperado, avísame y usamos un parser más robusto (`csv.reader`).


In [2]:
with open(VDEM_PATH, "r", encoding="utf-8") as f:
    n_filas = sum(1 for _ in f) - 1  # -1 por la cabecera

print(f"Número total de filas: {n_filas}")


Número total de filas: 28092


## Bloque 3 — Carga completa del CSV

**Objetivo:** cargar el CSV completo en un DataFrame de pandas.


**Errores habituales:** `MemoryError` o kernel muerto (RAM insuficiente) → avisar, no reintentar; `DtypeWarning` por columnas con tipos mixtos (mitigado con `low_memory=False`, y no es crítico en esta fase exploratoria).


In [3]:
t0 = time.time()
df_vdem = pd.read_csv(VDEM_PATH, low_memory=False)
t1 = time.time()

print(f"Shape: {df_vdem.shape}")
print(f"Tiempo de carga: {t1 - t0:.1f} segundos")
print(f"Memoria usada: {df_vdem.memory_usage(deep=True).sum() / 1e6:.1f} MB")


Shape: (28092, 4618)
Tiempo de carga: 15.7 segundos
Memoria usada: 1047.4 MB


## Bloque 4 — Perfilado diferenciado por columna (núcleo del análisis)

**Objetivo:** para cada una de las columnas, calcular tipo de dato, % de missing, nº de valores únicos y, según el tipo de columna, o bien el listado de valores (categóricas/identificatorias) o bien el rango (numéricas continuas).


- **Identificatorias/estructurales** (`country_name`, `year`, `COWcode`...): se listan sus valores únicos completos.
- **Numéricas continuas** (más de `UMBRAL_CATEGORICA` valores únicos): se reporta min/max/media/%missing.
- **Categóricas/ordinales** (pocos valores únicos, típico de `_ord`, dummies, o variables 0/1): se listan sus valores.

El umbral de 20 valores únicos es una decisión metodológica abierta — es razonable para distinguir escalas ordinales V-Dem (normalmente 3-6 niveles) de índices continuos (0-1 con muchos decimales), pero es ajustable si al revisar los resultados ves casos mal clasificados.

**Resultado esperado:** un DataFrame `df_metadata_vdem` con una fila por columna del CSV original, y un conteo de cuántas columnas caen en cada tipo.

**Qué comprobar:** que `df_metadata_vdem.shape[0]` sea igual al número de columnas del bloque 1; revisa `df_metadata_vdem['tipo_columna'].value_counts()` — si ves variables que sabes que son continuas (p.ej. `v2x_polyarchy`) clasificadas como categóricas, o viceversa, es señal de que `UMBRAL_CATEGORICA` necesita ajuste.

**Errores habituales:** este bloque puede tardar varios minutos dado el volumen de columnas — es normal, no lo interrumpas salvo que exceda ~10-15 minutos.


In [4]:
UMBRAL_CATEGORICA = 20  # nº máximo de valores únicos para tratar una columna numérica como categórica/ordinal

columnas_identificatorias = [
    'country_name', 'country_text_id', 'country_id', 'year', 'historical_date',
    'project', 'historical', 'histname', 'codingstart', 'codingend',
    'codingstart_contemp', 'codingend_contemp', 'codingstart_hist', 'codingend_hist',
    'gapstart1', 'gapstart2', 'gapstart3', 'gapend1', 'gapend2', 'gapend3',
    'gap_index', 'COWcode'
]

resumen = []

for col in df_vdem.columns:
    serie = df_vdem[col]
    n_unique = serie.nunique(dropna=True)
    pct_missing = serie.isna().mean() * 100
    dtype = str(serie.dtype)

    if col in columnas_identificatorias:
        tipo = 'identificatoria'
    elif pd.api.types.is_numeric_dtype(serie):
        tipo = 'categorica_ordinal' if n_unique <= UMBRAL_CATEGORICA else 'numerica_continua'
    else:
        tipo = 'categorica_texto'

    fila = {
        'columna': col,
        'dtype': dtype,
        'tipo_columna': tipo,
        'n_unique': n_unique,
        'pct_missing': round(pct_missing, 2),
    }

    if tipo == 'numerica_continua':
        fila['min'] = serie.min()
        fila['max'] = serie.max()
        fila['mean'] = round(serie.mean(), 4)
        fila['valores_unicos'] = np.nan
    else:
        valores = sorted(serie.dropna().unique().tolist(), key=str)[:50]
        fila['min'] = np.nan
        fila['max'] = np.nan
        fila['mean'] = np.nan
        fila['valores_unicos'] = str(valores)

    resumen.append(fila)

df_metadata_vdem = pd.DataFrame(resumen)

print(f"Total columnas perfiladas: {df_metadata_vdem.shape[0]}")
print(df_metadata_vdem['tipo_columna'].value_counts())
df_metadata_vdem.head(20)


Total columnas perfiladas: 4618
tipo_columna
numerica_continua     3143
categorica_ordinal    1428
categorica_texto        25
identificatoria         22
Name: count, dtype: int64


,columna,dtype,tipo_columna,n_unique,pct_missing,min,max,mean,valores_unicos
0,country_name,str,identificatoria,202,0.00,NaN,NaN,NaN,"['Afghanistan', 'Albania', 'Algeria', 'Angola'..."
1,country_text_id,str,identificatoria,202,0.00,NaN,NaN,NaN,"['AFG', 'AGO', 'ALB', 'ARE', 'ARG', 'ARM', 'AU..."
2,country_id,int64,identificatoria,202,0.00,NaN,NaN,NaN,"[10, 100, 101, 102, 103, 104, 105, 106, 107, 1..."
3,year,int64,identificatoria,237,0.00,NaN,NaN,NaN,"[1789, 1790, 1791, 1792, 1793, 1794, 1795, 179..."
4,historical_date,str,identificatoria,237,0.00,NaN,NaN,NaN,"['1789-12-31', '1790-12-31', '1791-12-31', '17..."
5,project,int64,identificatoria,3,0.00,NaN,NaN,NaN,"[0, 1, 2]"
6,historical,int64,identificatoria,2,0.00,NaN,NaN,NaN,"[0, 1]"
7,histname,str,identificatoria,814,0.36,NaN,NaN,NaN,"['Aden, part of British India', 'Afghan Transi..."
8,codingstart,int64,identificatoria,47,0.00,NaN,NaN,NaN,"[1789, 1798, 1800, 1802, 1804, 1806, 1809, 181..."
9,codingend,int64,identificatoria,11,0.00,NaN,NaN,NaN,"[1859, 1860, 1861, 1866, 1867, 1870, 1871, 194..."


## Bloque 5 — Agrupación por variable base (variantes técnicas)

**Objetivo:** identificar, para cada columna, su "variable base" quitando sufijos técnicos (`_codelow`, `_codehigh`, `_sd`, `_osp`, `_ord`, `_mean`, `_nr`), y marcar por separado las columnas dummy de categoría múltiple (terminadas en `_0`, `_1`, `_2`...).


**Resultado esperado:** una columna nueva `variable_base` en `df_metadata_vdem`, una columna booleana `es_dummy_categoria`, y un conteo de variables base únicas. En función de lo visto en el codebook, deberíamos esperar entre 814 y 870 variables base

**Qué comprobar:** toma 3-4 ejemplos al azar (p.ej. `v2x_polyarchy_codelow`, `v2eldonate_osp_codelow`) y verifica manualmente que `variable_base` sea la que esperarías. Si ves casos raros (p.ej. variables que terminan de forma ambigua), coméntamelos — la función es heurística, no infalible.

**Errores habituales:** ninguno esperado; es una operación de texto sobre nombres de columnas ya cargados.


In [5]:
suffix_pattern = re.compile(r'(_codelow|_codehigh|_sd|_osp|_ord|_mean|_nr)$')
dummy_pattern = re.compile(r'_\d+$')

def obtener_variable_base(col):
    base = col
    while True:
        m = suffix_pattern.search(base)
        if m:
            base = base[:m.start()]
        else:
            break
    return base

def es_dummy_categoria(col):
    return bool(dummy_pattern.search(col))

df_metadata_vdem['variable_base'] = df_metadata_vdem['columna'].apply(obtener_variable_base)
df_metadata_vdem['es_dummy_categoria'] = df_metadata_vdem['columna'].apply(es_dummy_categoria)

n_variables_base = df_metadata_vdem.loc[~df_metadata_vdem['es_dummy_categoria'], 'variable_base'].nunique()

print(f"Nº de columnas totales: {df_metadata_vdem.shape[0]}")
print(f"Nº de variables base (agrupando variantes tecnicas, sin contar dummies): {n_variables_base}")

# Ejemplo de verificacion manual: variantes agrupadas bajo una misma variable base
df_metadata_vdem[df_metadata_vdem['variable_base'] == 'v2x_polyarchy'][['columna', 'variable_base', 'tipo_columna']]


Nº de columnas totales: 4618
Nº de variables base (agrupando variantes tecnicas, sin contar dummies): 856


,columna,variable_base,tipo_columna
22,v2x_polyarchy,v2x_polyarchy,numerica_continua
23,v2x_polyarchy_codelow,v2x_polyarchy,numerica_continua
24,v2x_polyarchy_codehigh,v2x_polyarchy,numerica_continua
25,v2x_polyarchy_sd,v2x_polyarchy,numerica_continua


Muchas columnas del CSV terminan en un número (por ejemplo v2eltype_0, v2eltype_1, v2casoe_3) porque V-Dem codifica ciertas preguntas de opción múltiple como una serie de columnas binarias — una por cada respuesta posible — en vez de una sola columna categórica. Estas columnas dummy no tienen sufijos técnicos como _sd u _ord, así que la lógica del Bloque 5 las trata, por sí sola, como si fueran "variable base" (porque su nombre no cambia al quitarles sufijos). Para evitar confundir estas dummies con variables base reales, es_variable_base_real combina ambas condiciones: solo es True cuando la columna no tiene sufijo técnico recortado y tampoco es una dummy de categoría. Así, de las 4.618 columnas totales, quedan aisladas 834 que son efectivamente el punto de partida (una fila = una variable conceptual) para elegir qué incorporar al bloque institucional 

In [6]:
df_metadata_vdem['es_columnabase'] = df_metadata_vdem['columna'] == df_metadata_vdem['variable_base']
df_metadata_vdem['es_variable_base_real'] = df_metadata_vdem['es_columnabase'] & ~df_metadata_vdem['es_dummy_categoria']

## Bloque 6 — Missingness: ranking y distribución

**Objetivo:** identificar qué columnas (y qué variables base) tienen mayor proporción de datos faltantes.

**Justificación metodológica:** esto es solo una vista descriptiva en esta fase exploratoria 

**Resultado esperado:** una tabla ordenada por `pct_missing` descendente, y un histograma de la distribución de `pct_missing` en todas las columnas.

**Qué comprobar:** si ves columnas identificatorias con missing alto (no deberían tenerlo, salvo `historical_date` u otras opcionales), es señal de revisar. Presta atención a si el patrón de missingness concentra en columnas asociadas a periodos históricos (`v3*`) frente a las contemporáneas (`v2*`) — es esperable dado que V-Dem amplía cobertura temporal con el proyecto histórico.

**Errores habituales:** ninguno esperado.


In [7]:
missingness_ranking = df_metadata_vdem[['columna', 'variable_base', 'tipo_columna', 'pct_missing']]\
    .sort_values('pct_missing', ascending=False)

print(missingness_ranking.head(30))

fig_missing = px.histogram(
    df_metadata_vdem,
    x='pct_missing',
    nbins=50,
    title='Distribucion de % de missing por columna (V-Dem v16)',
    labels={'pct_missing': '% de valores faltantes'}
)
figuras_reporte.append(fig_missing)
fig_missing.show()


          columna variable_base        tipo_columna  pct_missing
2943   v3elupvtlg    v3elupvtlg  categorica_ordinal        99.99
2944   v3elupvtsm    v3elupvtsm  categorica_ordinal        99.99
2948  v3eltvriguc   v3eltvriguc  categorica_ordinal        99.94
2938    v3elageuc     v3elageuc  categorica_ordinal        99.93
2947   v3elupstsm    v3elupstsm   numerica_continua        99.74
2945   v3elupstsl    v3elupstsl   numerica_continua        99.74
2946   v3elupseat    v3elupseat   numerica_continua        99.73
3314   v3elvotsml    v3elvotsml   numerica_continua        99.71
3321   v3elvotlrg    v3elvotlrg   numerica_continua        99.50
3326    v3eltvrig     v3eltvrig  categorica_ordinal        99.42
16      gapstart3     gapstart3     identificatoria        99.39
19        gapend3       gapend3     identificatoria        99.39
2951    v3elagepr     v3elagepr  categorica_ordinal        99.26
3322   v3ellovtsm    v3ellovtsm   numerica_continua        99.24
2941   v3eldirepr    v3el

## Bloque 7 — Exportar metadata a Excel

**Objetivo:** guardar `df_metadata_vdem` como archivo Excel, replicando el patrón de `metadata.xlsx` (WDI) ya usado en el proyecto.

**Justificación metodológica:** este archivo es el output que usaremos como punto de partida para decidir, en la siguiente fase, qué variables base de V-Dem se seleccionan para el bloque institucional del TFM — combinando esta evidencia empírica con el criterio teórico del codebook.

**Resultado esperado:** un archivo `metadata_vdem_v16.xlsx` en tu directorio de trabajo local.

**Qué comprobar:** ábrelo y revisa que las ~50 primeras filas (columnas identificatorias) tengan sentido, y que el resto muestre `min`/`max` para las continuas y `valores_unicos` para las categóricas.

**Errores habituales:** si el Excel falla por tamaño de celda (columnas de texto muy largas en `valores_unicos`), lo trabajamos en el siguiente paso — no debería pasar porque ya limitamos a 50 valores por celda en el Bloque 4.


In [8]:
df_metadata_vdem.to_excel("metadata_vdem.xlsx", index=False)
print("Metadata exportada a metadata_vdem.xlsx")


Metadata exportada a metadata_vdem.xlsx


## INDICES INTERMEDIOS

Luego de estudiar el codebook que acompaña la base del VDEM, se decidió trabajar con 79 componentes que relevan elementos conceptuales de interés para este trabajo (10 de la Sección 2.2 + 69 de otras secciones).  

In [9]:
indices_intermedios_seleccionados = [
    'v2x_clphy', 'v2x_cspart', 'v2x_delibdem_stock', 'v2x_diagacc', 'v2x_divparctrl',
    'v2x_egaldem_stock', 'v2x_elecreg', 'v2x_electoral_integrity', 'v2x_ex_confidence',
    'v2x_ex_direlect', 'v2x_ex_hereditary', 'v2x_ex_military', 'v2x_ex_party',
    'v2x_execorr', 'v2x_feduni', 'v2x_gencl', 'v2x_gencs', 'v2x_genpp', 'v2x_horacc',
    'v2x_hosinter', 'v2x_jucon', 'v2x_libdem_stock', 'v2x_partipdem_stock',
    'v2x_polyarchy_stock', 'v2x_pubcorr', 'v2x_regime_amb', 'v2x_suffr', 'v2xca_academ',
    'v2xcl_acjst', 'v2xcl_disc', 'v2xcl_dmove', 'v2xcl_prpty', 'v2xcl_slave',
    'v2xcs_ccsi', 'v2xdd_dd', 'v2xdd_i_ci', 'v2xdd_i_or', 'v2xdd_i_pl', 'v2xdd_i_rf',
    'v2xdl_delib', 'v2xed_ed_cent', 'v2xed_ed_con', 'v2xed_ed_ctag', 'v2xed_ed_dmcon',
    'v2xed_ed_poed', 'v2xed_ed_ptcon', 'v2xed_ptcon', 'v2xedvd_me_cent', 'v2xedvd_me_ctag',
    'v2xeg_eqaccess', 'v2xeg_eqdr', 'v2xel_elecparl', 'v2xel_elecpres', 'v2xel_locelec',
    'v2xel_regelec', 'v2xex_elecleg', 'v2xlg_legcon', 'v2xlg_leginter', 'v2xme_altinf',
    'v2xnp_pres', 'v2xnp_regcorr', 'v2xpas_democracy', 'v2xpas_democracy_government',
    'v2xpas_democracy_opposition', 'v2xpas_economic', 'v2xpas_economic_government',
    'v2xpas_economic_opposition', 'v2xpas_exclusion', 'v2xpas_exclusion_government',
    'v2xpas_exclusion_opposition', 'v2xpas_religion', 'v2xpas_religion_government',
    'v2xpas_religion_opposition', 'v2xpe_exlecon', 'v2xpe_exlgender', 'v2xpe_exlgeo',
    'v2xpe_exlpol', 'v2xpe_exlsocgr', 'v2xps_party'
]

print(f"Total de indices seleccionados: {len(indices_intermedios_seleccionados)}")

df_metadata_vdem_indices_intermedios = df_metadata_vdem[
    df_metadata_vdem['columna'].isin(indices_intermedios_seleccionados)
].copy()

print(f"Filas en df_metadata_vdem_indices_intermedios: {df_metadata_vdem_indices_intermedios.shape[0]}")
df_metadata_vdem_indices_intermedios

Total de indices seleccionados: 79
Filas en df_metadata_vdem_indices_intermedios: 79


,columna,dtype,tipo_columna,n_unique,pct_missing,min,max,mean,valores_unicos,variable_base,es_dummy_categoria,es_columnabase,es_variable_base_real
58,v2x_suffr,float64,numerica_continua,72,0.60,0.000,1.000,0.4977,NaN,v2x_suffr,False,True,True
72,v2x_jucon,float64,numerica_continua,979,5.55,0.003,0.991,0.4965,NaN,v2x_jucon,False,True,True
76,v2xlg_legcon,float64,numerica_continua,957,22.18,0.012,0.989,0.4440,NaN,v2xlg_legcon,False,True,True
84,v2x_cspart,float64,numerica_continua,953,2.23,0.014,0.987,0.3694,NaN,v2x_cspart,False,True,True
88,v2xdd_dd,float64,numerica_continua,494,29.78,0.000,0.781,0.0476,NaN,v2xdd_dd,False,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4030,v2xed_ed_dmcon,float64,numerica_continua,706,61.97,0.019,0.989,0.4593,NaN,v2xed_ed_dmcon,False,True,True
4034,v2xed_ed_ptcon,float64,numerica_continua,552,61.56,0.005,0.960,0.5115,NaN,v2xed_ed_ptcon,False,True,True
4038,v2xed_ptcon,float64,numerica_continua,664,61.93,0.002,0.972,0.4903,NaN,v2xed_ptcon,False,True,True
4050,v2xedvd_me_cent,float64,numerica_continua,939,54.82,0.012,0.981,0.5125,NaN,v2xedvd_me_cent,False,True,True


In [10]:
df_metadata_vdem_indices_intermedios = df_metadata_vdem_indices_intermedios.reset_index(drop=True)

df_metadata_vdem_indices_intermedios

,columna,dtype,tipo_columna,n_unique,pct_missing,min,max,mean,valores_unicos,variable_base,es_dummy_categoria,es_columnabase,es_variable_base_real
0,v2x_suffr,float64,numerica_continua,72,0.60,0.000,1.000,0.4977,NaN,v2x_suffr,False,True,True
1,v2x_jucon,float64,numerica_continua,979,5.55,0.003,0.991,0.4965,NaN,v2x_jucon,False,True,True
2,v2xlg_legcon,float64,numerica_continua,957,22.18,0.012,0.989,0.4440,NaN,v2xlg_legcon,False,True,True
3,v2x_cspart,float64,numerica_continua,953,2.23,0.014,0.987,0.3694,NaN,v2x_cspart,False,True,True
4,v2xdd_dd,float64,numerica_continua,494,29.78,0.000,0.781,0.0476,NaN,v2xdd_dd,False,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
74,v2xed_ed_dmcon,float64,numerica_continua,706,61.97,0.019,0.989,0.4593,NaN,v2xed_ed_dmcon,False,True,True
75,v2xed_ed_ptcon,float64,numerica_continua,552,61.56,0.005,0.960,0.5115,NaN,v2xed_ed_ptcon,False,True,True
76,v2xed_ptcon,float64,numerica_continua,664,61.93,0.002,0.972,0.4903,NaN,v2xed_ptcon,False,True,True
77,v2xedvd_me_cent,float64,numerica_continua,939,54.82,0.012,0.981,0.5125,NaN,v2xedvd_me_cent,False,True,True


## METADATA de 79 indices intermedios

### Extrayendo texto del pdf

Como la metadata de los indices solo esta en el pdf codebook, vamos a extraer el texto y nombre de los indicadores intermedios para agregar columnas descriptivas que aporten interpretabilidad

Objetivo: leer el codebook (PDF) y extraer todo su texto como líneas, para poder buscar en él la definición de cada una de las 79 variables

Vamos a cargar el PDF

In [11]:
!pip install pypdf


[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
from pypdf import PdfReader

CODEBOOK_PATH = "codebook.pdf"  # ajusta si esta en otra carpeta

reader = PdfReader(CODEBOOK_PATH)
texto_completo = "\n".join(pagina.extract_text() or "" for pagina in reader.pages)
lineas_codebook = texto_completo.split("\n")

print(f"Numero de paginas: {len(reader.pages)}")
print(f"Numero de lineas de texto extraidas: {len(lineas_codebook)}")

Numero de paginas: 507
Numero de lineas de texto extraidas: 25892


Extraer nombre, pregunta operativa y descripción (clarification) para las 79 variables

In [13]:
import re

def extraer_metadata_variable(variable, lineas):
    patron_header = re.compile(r'\(' + re.escape(variable) + r'\)\s*$')
    idx_header = None
    for i, linea in enumerate(lineas):
        if patron_header.search(linea.strip()):
            idx_header = i
            break
    if idx_header is None:
        return None, None, None

    linea_header = lineas[idx_header].strip()
    match_nombre = re.match(
        r'^[\d.]*\s*(.+?)\s*\([ABCD]\*?\)\s*\(' + re.escape(variable) + r'\)$',
        linea_header
    )
    nombre = match_nombre.group(1).strip() if match_nombre else None

    bloque = lineas[idx_header: idx_header + 40]
    texto_bloque = "\n".join(bloque)

    def extraer_campo(campo, texto):
        patron = re.compile(
            campo + r':\s*(.*?)(?=\n\s*(Clarification|Responses|Scale|Note|'
            r'Additional versions|Source\(s\)|Data release|Aggregation|'
            r'Cross-coder|Country|Citation):|\Z)',
            re.DOTALL
        )
        m = patron.search(texto)
        if m:
            return re.sub(r'\s+', ' ', m.group(1)).strip()
        return None

    question = extraer_campo('Question', texto_bloque)
    clarification = extraer_campo('Clarification', texto_bloque)
    return nombre, question, clarification

nombres, questions, descriptions = [], [], []
for var in df_metadata_vdem_indices_intermedios['columna']:
    nombre, question, clarification = extraer_metadata_variable(var, lineas_codebook)
    nombres.append(nombre)
    questions.append(question)
    descriptions.append(clarification)

df_metadata_vdem_indices_intermedios['nombre'] = nombres
df_metadata_vdem_indices_intermedios['question'] = questions
df_metadata_vdem_indices_intermedios['description'] = descriptions

print(f"Sin nombre: {df_metadata_vdem_indices_intermedios['nombre'].isna().sum()}")
print(f"Sin question: {df_metadata_vdem_indices_intermedios['question'].isna().sum()}")
print(f"Sin description: {df_metadata_vdem_indices_intermedios['description'].isna().sum()}")

df_metadata_vdem_indices_intermedios[['columna', 'nombre', 'question', 'description']]

Sin nombre: 11
Sin question: 5
Sin description: 10


,columna,nombre,question,description
0,v2x_suffr,Share of population with suffrage,What share of adult citizens as defined by sta...,This question does not take into consideration...
1,v2x_jucon,Judicial constraints on the executive index,To what extent does the executive respect the ...,NaN
2,v2xlg_legcon,Legislative constraints on the executive index,To what extent are the legislature and governm...,The participatory principle of democracy empha...
3,v2x_cspart,Civil society participation index,Are major CSOs routinely consulted by policyma...,The sphere of civil society lies in the public...
4,v2xdd_dd,Direct popular vote index,To what extent is the direct popular vote util...,Direct popular voting refers here to an instit...
...,...,...,...,...
74,v2xed_ed_dmcon,Democratic indoctrination content in education,To what extent is the indoctrination content i...,Indoctrination content in education can range ...
75,v2xed_ed_ptcon,Patriotic indoctrination content in education,To what extent is the indoctrination content i...,Patriotism is another key tool that regimes ca...
76,v2xed_ptcon,Patriotic indoctrination content in education ...,To what extent is the indoctrination content i...,This is an aggregate index of patriotic indoct...
77,v2xedvd_me_cent,Centralization of media control,Is control over the media centralized?,This index measures the extent to which the me...


Aquí voy a exportar esta metadata para verificar manualmente que los nombres y descripciones se hayan asignado correctamente

In [14]:
df_metadata_vdem_indices_intermedios[["columna", "nombre", "question", "description"]].to_excel(
    "metadatavdem_79_indices_intermedios.xlsx", index=False
)
print("Metadata exportada a metadatavdem_79_indices_intermedios.xlsx")

Metadata exportada a metadatavdem_79_indices_intermedios.xlsx


### Validación y corrección manual
Algunos nombres, preguntas y descripciones fueron mal asignados o quedaron como campos vacíos. Se validó y asignó manualmente de forma correcta, siguiendo el pdf. 



In [15]:
nombres_manuales = {
    'v2x_regime_amb': "Regimes of the world - the RoW measure with categories for ambiguous cases",
    'v2xdd_i_or': "Obligatory referendum index",
    'v2x_electoral_integrity': "Electoral Integrity Index",
    'v2x_polyarchy_stock': "Electoral Democracy Stock",
    'v2x_partipdem_stock': "Participatory Democracy Stock",
    'v2x_delibdem_stock': "Deliberative Democracy Stock",
    'v2x_egaldem_stock': "Egalitarian Democracy Stock",
    'v2xpas_democracy_opposition': "Opposition Parties' Democracy Index",
    'v2xpas_religion_government': "Government Coalition Religion Index",
    'v2xpas_exclusion_opposition': "Opposition Parties' Exclusion Index",
    'v2xpas_economic_government': "Government Coalition Left-Right Index",
}

for var, nombre in nombres_manuales.items():
    mask = df_metadata_vdem_indices_intermedios['columna'] == var
    df_metadata_vdem_indices_intermedios.loc[mask, 'nombre'] = nombre

print(f"Sin nombre despues de la asignacion manual: {df_metadata_vdem_indices_intermedios['nombre'].isna().sum()}")

Sin nombre despues de la asignacion manual: 0


Manualmente revisamos preguntas y descripciones faltantes o mal escritas y las corregimos a continuación

In [16]:
# PREGUNTAS
questions_manuales = {
    'v2x_electoral_integrity': "To what extent is the ideal of electoral integrity achieved?",
    'v2x_polyarchy_stock': "What is the country's accumulated stock of electoral democracy?",
    'v2x_partipdem_stock': "What is the country's accumulated stock of participatory democracy?",
    'v2x_delibdem_stock': "What is the country's accumulated stock of deliberative democracy?",
    'v2x_egaldem_stock': "What is the country's accumulated stock of egalitarian democracy?",
    # v2x_libdem_stock quedaba con el encabezado de pagina "TOC 349 ..." pegado al final de la pregunta
    'v2x_libdem_stock': "What is the country's accumulated stock of liberal democracy?",
    'v2xlg_legcon': "To what extent are the legislature and government agencies e.g., comptroller general, general prosecutor, or ombudsman capable of questioning, investigating, and exercising oversight over the executive?",
}

for var, question in questions_manuales.items():
    mask = df_metadata_vdem_indices_intermedios['columna'] == var
    df_metadata_vdem_indices_intermedios.loc[mask, 'question'] = question

print(f"Sin question despues de la asignacion manual: {df_metadata_vdem_indices_intermedios['question'].isna().sum()}")

Sin question despues de la asignacion manual: 0


Para descripcion detectamos que algunas estaban mal asignadas y faltaban otras. Esto se arregla manualmente a continuación. Además hay 11 que en el codebook efectivamente no tienen descripcion, así que estas es correcto que figuren vacías (a las 8 originales se suman v2xlg_legcon, v2xcl_acjst y v2x_hosinter, detectadas en una segunda auditoría fila por fila contra el codebook).

In [17]:
import re

# 1. Vaciar las descriptions con contenido incorrecto (pertenecen a otro indice, no tienen Clarification propio en el codebook)
# v2xlg_legcon, v2xcl_acjst y v2x_hosinter se suman tras una segunda auditoria fila por fila contra el codebook
sin_clarification_real = ['v2xel_elecpres', 'v2xlg_leginter', 'v2xme_altinf', 'v2xlg_legcon', 'v2xcl_acjst', 'v2x_hosinter']
mask = df_metadata_vdem_indices_intermedios['columna'].isin(sin_clarification_real)
df_metadata_vdem_indices_intermedios.loc[mask, 'description'] = None

# 2. Limpiar la contaminacion "TOC ... <titulo de capitulo/seccion>" del resto
patron_toc = re.compile(r'TOC \d+ .*?(?=\s[A-Z][a-z])')

def limpiar_toc(texto):
    if pd.isna(texto):
        return texto
    return re.sub(r'TOC \d+.*?\d{2,3}\.\d{1,2}(\.\d{1,2})?\s+[A-Za-z\'’\- ]+', ' ', texto).strip()

df_metadata_vdem_indices_intermedios['description'] = df_metadata_vdem_indices_intermedios['description'].apply(limpiar_toc)

print("Sin description ahora:", df_metadata_vdem_indices_intermedios['description'].isna().sum())

Sin description ahora: 16


Vamos a corregir algunas "descriptions". No es que estén mal tomadas, sino que muchas de ellas tienen saltos de página o componentes que entorpecen la lectura

In [18]:
descriptions_manuales = {
    'v2x_electoral_integrity': "Electoral integrity is a set of principles to be achieved in elections which empower the everyday citizen and help to realize the ideals of democracy. There are four principles of electoral integrity: 1) Contestation; 2) Participation; 3) Deliberation; and 4) Adjudication.",
    'v2x_delibdem_stock': "A country's accumulated stock of deliberative democracy accounts for the cumulative sum of all past values on the Deliberative Democracy Index, with a one-percent annual depreciation rate. This measure reflects the extent to which a country has accumulated experience over time with deliberative democracy, compared to a hypothetical country that has always attained the ideal of deliberative democracy to the fullest extent. A higher value indicates longer or more consistent deliberative democracy, even if recent levels have fluctuated.",
    'v2x_egaldem_stock': "A country's accumulated stock of egalitarian democracy accounts for the cumulative sum of all past values on the Egalitarian Democracy Index, with a one-percent annual depreciation rate. This measure reflects the extent to which a country has accumulated experience over time with egalitarian democracy, compared to a hypothetical country that has always attained the ideal of egalitarian democracy to the fullest extent. A higher value indicates longer or more consistent egalitarian democracy, even if recent levels have fluctuated.",
    'v2x_partipdem_stock': "A country's accumulated stock of participatory democracy accounts for the cumulative sum of all past values on the Participatory Democracy Index, with a one-percent annual depreciation rate. This measure reflects the extent to which a country has accumulated experience over time with participatory democracy, compared to a hypothetical country that has always attained the ideal of participatory democracy to the fullest extent. A higher value indicates longer or more consistent participatory democracy, even if recent levels have fluctuated.",
    'v2x_polyarchy_stock': "A country's accumulated stock of electoral democracy accounts for the cumulative sum of all past values on the Electoral Democracy Index, with a one-percent annual depreciation rate. This measure reflects the extent to which a country has accumulated experience over time with electoral democracy, compared to a hypothetical country that has always attained the ideal of electoral democracy to the fullest extent. A higher value indicates longer or more consistent electoral democracy, even if recent levels have fluctuated.",
    # v2x_ex_party y v2xps_party quedaban con el fragmento "TOC NN ... <titulo capitulo>" pegado a mitad de frase
    # (el codebook corta la pagina justo en medio del parrafo de Clarification); se completan con el texto limpio del codebook
    'v2x_ex_party': "Representing one of five regime dimensions, each of which may be more or less present in any given case, this index taps into the extent to which a ruling party appoints and dismisses the chief executive.",
    'v2xps_party': "Party institutionalization refers to various attributes of the political parties in a country, e.g., level and depth of organization, links to civil society, cadres of party activists, party supporters within the electorate, coherence of party platforms and ideologies, party-line voting among representatives within the legislature. A high score on these attributes generally indicates a more institutionalized party system. This index considers the attributes of all parties with an emphasis on larger parties, i.e., those that may be said to dominate and define the party system.",
    # v2xpas_exclusion_opposition tenia el titulo de la seccion siguiente (6.10 Party-System Left-Right Index) pegado a mitad de frase
    'v2xpas_exclusion_opposition': "The Opposition Parties' Exclusion Index (OPEXI) ranges from 0 to 1, where lower values are associated with more inclusive opposition parties and higher values with opposition parties advocating for more exclusion. As this index is calculated for country-election-year, we advise caution using it for years where a country does not have a general election (lower house).",
    # Las siguientes 8 tenian contaminacion de "TOC NN ..." (salto de pagina) o palabras pegadas por el mismo motivo,
    # detectadas en una auditoria exhaustiva letra por letra contra el codebook
    'v2x_suffr': "This question does not take into consideration restrictions based on age, residence, having been convicted for crime, or being legally incompetent. It covers legal de jure restrictions, not restrictions that may be operative in practice de facto. The adult population as defined by statute is defined by citizens in the case of independent countries or the people living in the territorial entity in the case of colonies. Universal suffrage is coded as 100%. Universal male suffrage only is coded as 50%. Years before electoral provisions are introduced are scored 0%. The scores do not reflect whether an electoral regime was interrupted or not. Only if new constitutions, electoral laws, or the like explicitly introduce new regulations of suffrage, the scores were adjusted accordingly if the changes suggested doing so. If qualifying criteria other than gender apply such as property, tax payments, income, literacy, region, race, ethnicity, religion, and/or 'economic independence', estimates have been calculated by combining information on the restrictions with different kinds of statistical information on population size, age distribution, wealth distribution, literacy rates, size of ethnic groups, etc., secondary country-specific sources, and — in the case of very poor information — the conditions in similar countries or colonies. The scores reflect de jure provisions of suffrage extension in percentage of the adult population. If the suffrage law is revised in a way that affects the extension, the scores reflect this change as of the calendar year the law was enacted.",
    'v2x_cspart': "The sphere of civil society lies in the public space between the private sphere and the state. Here, citizens organize in groups to pursue their collective interests and ideals. We call these groups civil society organizations CSOs. CSOs include, but are by no means limited to, interest groups, labor unions, spiritual organizations if they are engaged in civic or political activities, social movements, professional associations, charities, and other non-governmental organizations. The core civil society index CCSI is designed to provide a measure of a robust civil society, understood as one that enjoys autonomy from the state and in which citizens freely and actively pursue their political and civic goals, however conceived.",
    'v2x_elecreg': "Coded 0 until an executive or legislative election is held, defined by v2xel_elecpres and v2xel_elecparl, then set to 1 until any of the following two events occur (if they occur): (a) that the election was \"aborted\", meaning that those elected did not resume power, as defined by v2x_hosabort and v2x_legabort; or (b) an \"electoral interruption\", meaning that either the legislature was shut down, as defined by v2xlg_leginter, or there was an executive coup, as defined by v2x_hosinter; in the case of (a) or (b), v2x_elecreg is set to 0 until there is another election. The operational indicator of an \"aborted\" executive election (v2x_hosabort) is that v2expathhs did not turn 7 within 12 months after the election, for a legislative election (v2x_legabort) that v2lgbicam did not turn positive within 12 months after the election. An interruption of the electoral regime occurring through the HOS, e.g. a coup d'etat, is indicated by v2x_hosinter as a change in v2xel_elecpres, meaning v2expathhs turned from 7 to something else, with the exception of 6, approval by the legislature (in case the legislature remained in place). An interruption of the electoral regime occurring through the legislature is defined by v2xlg_leginter based on v2lgbicam turning 0. We note that the coding of v2x_elecreg does not merely follow mechanically from the scores on these other variables, as the coding of v2x_elecreg has also been cross-checked and validated by research assistants. An executive and a legislative electoral regime cannot be separated since they form an integral part, where an aborted legislature is interpreted as a signal that also the executive is not standing for election any longer, and vice versa.",
    'v2xcl_slave': "Involuntary servitude occurs when an adult is unable to quit a job s/he desires to leave — not by reason of economic necessity but rather by reason of employer's coercion. This includes labor camps but not work or service which forms part of normal civic obligations such as conscription or employment in command economies.",
    'v2xex_elecleg': "If the legislature is unicameral, v2xex_elecleg is measured as the proportion of legislators directly elected + half of the proportion that are indirectly elected. If the legislature is bicameral and the upper house is involved in the appointment of the chief executive, the same proportion of directly and half of the indirectly elected legislators is calculated for the upper house; the scores for the lower and upper houses are then averaged. Note that a popular election is minimally defined and also includes sham elections with limited suffrage and no competition. This index is useful primarily for aggregating higher-order indices and should not necessarily be interpreted as an important element of democracy in its own right. Since the variables coding the share of directly and indirectly elected legislators are not yet fully in sync for all country dates, a few observations now receive an index value larger than 1.",
    'v2xeg_eqaccess': "The Equal Access subcomponent is based on the idea that neither the protections of rights and freedoms nor the equal distribution of resources is sufficient to ensure adequate representation. Ideally, all groups should enjoy equal de facto capabilities to participate, to serve in positions of political power, to put issues on the agenda, and to influence policymaking.",
    'v2xcl_dmove': "This indicator specifies the extent to which citizens are able to move freely, in daytime and nighttime, in public thoroughfares, across regions within a country, and to establish permanent residency where they wish. Note that restrictions in movement might be imposed by the state and/or by informal norms and practices. Such restrictions sometimes fall on rural residents, on specific social groups, or on dissidents. Do not consider restrictions in movement that are placed on ordinary non-political criminals. Do not consider restrictions in movement that result from crime or unrest.",
    'v2x_horacc': "Horizontal accountability concerns the power of state institutions to oversee the government by demanding information, questioning officials and punishing improper behavior. This form of accountability ensures checks between institutions and prevents the abuse of power. The key agents in horizontal government accountability are: the legislature; the judiciary; and specific oversight agencies such as ombudsmen, prosecutor and comptroller generals.",
    # v2xeg_eqdr: un solo espacio faltante ("servicesi.e.means-tests"), detectado en la auditoria final
    'v2xeg_eqdr': "This component measures the extent to which resources — both tangible and intangible — are distributed in society. An equal distribution of resources supports egalitarian democracy in two ways. First, lower poverty rates and the distribution of goods and services such as food, water, housing, education and healthcare ensure that all individuals are capable of participating in politics and government. In short, basic needs must be met in order for individuals to effectively exercise their rights and freedoms see, for example, Sen 1999, Maslow 1943. Second, high levels of resource inequality undermine the ability of poorer populations to participate meaningfully Aristotle, Dahl 2006. Thus, it is necessary to include not only measures of poverty and the distribution of goods and services, but also the levels of inequality in these distributions, and the proportion of the population who are not eligible for social services i.e. means-tests, particularistic distribution, etc.. This principle also implies that social or economic inequalities can translate into political inequalities, an issue addressed most notably by Walzer 1983, who argues that overlapping \"spheres\" of inequality are particularly harmful to society. To address these overlapping \"spheres\", this component also includes measures of the distribution of power in society amongst different socio-economic groups, genders, etc.",
}

for var, desc in descriptions_manuales.items():
    mask = df_metadata_vdem_indices_intermedios['columna'] == var
    df_metadata_vdem_indices_intermedios.loc[mask, 'description'] = desc

# Estas 11 quedan vacias a proposito: el codebook no documenta Clarification para ellas
sin_clarification_confirmado = [
    'v2x_jucon', 'v2x_regime_amb', 'v2xdd_i_ci', 'v2xdd_i_rf', 'v2xdd_i_pl',
    'v2xel_elecpres', 'v2xlg_leginter', 'v2xme_altinf',
    'v2xlg_legcon', 'v2xcl_acjst', 'v2x_hosinter'
]

print("Sin description tras la correccion:", df_metadata_vdem_indices_intermedios['description'].isna().sum())
print("Deberian ser exactamente estas 11:", sorted(df_metadata_vdem_indices_intermedios[df_metadata_vdem_indices_intermedios['description'].isna()]['columna'].tolist()) == sorted(sin_clarification_confirmado))

Sin description tras la correccion: 11
Deberian ser exactamente estas 11: True


In [19]:
# 1. Corregir v2xdd_i_or (tenia question y description de otra variable)
mask = df_metadata_vdem_indices_intermedios['columna'] == 'v2xdd_i_or'
df_metadata_vdem_indices_intermedios.loc[mask, 'question'] = "To what extent is the obligatory referendum utilized?"
df_metadata_vdem_indices_intermedios.loc[mask, 'description'] = None  # no tiene Clarification en el codebook

# 2. Corregir espaciado de las 5 questions con palabras pegadas
questions_corregidas = {
    'v2xpas_exclusion': "To what extent does the party system reject cultural superiority and support immigration policies and the equal participation of women in the labor market?",
    'v2xed_ed_con': "To what extent is the indoctrination content in education democratic (and not patriotic)?",
    'v2xdd_i_ci': "To what extent is the popular initiative utilized?",
    'v2xdd_i_rf': "To what extent is the referendum utilized?",
    'v2xdd_i_pl': "To what extent is the plebiscite utilized?",
}

for var, question in questions_corregidas.items():
    mask = df_metadata_vdem_indices_intermedios['columna'] == var
    df_metadata_vdem_indices_intermedios.loc[mask, 'question'] = question


In [20]:
df_metadata_vdem_indices_intermedios[["columna", "nombre", "question", "description"]].to_excel(
    "metadatavdem_79_indices_intermedios.xlsx", index=False
)
print("Metadata exportada a metadatavdem_79_indices_intermedios.xlsx")

Metadata exportada a metadatavdem_79_indices_intermedios.xlsx


## Códigos de países de VDEM

Quiero obtener un df de valores unicos de las siguientes columnas: country_name, country_text_id, country_id, COWcode
para utilizar en la estandarización de codigos.

In [21]:
df_paises_vdem = df_vdem[['country_name', 'country_text_id', 'country_id', 'COWcode']] \
    .drop_duplicates() \
    .sort_values('country_name') \
    .reset_index(drop=True)

print(f"Filas (combinaciones unicas): {df_paises_vdem.shape[0]}")
df_paises_vdem

Filas (combinaciones unicas): 211


,country_name,country_text_id,country_id,COWcode
0,Afghanistan,AFG,36,700.0
1,Albania,ALB,12,339.0
2,Algeria,DZA,103,615.0
3,Angola,AGO,104,540.0
4,Argentina,ARG,37,160.0
...,...,...,...,...
206,Yemen,YEM,14,679.0
207,Zambia,ZMB,61,551.0
208,Zanzibar,ZZB,236,511.0
209,Zanzibar,ZZB,236,NaN


In [22]:
#Exportamos 
df_paises_vdem.to_excel("df_paises_vdem.xlsx", index=False)
print("Exportado a df_paises_vdem.xlsx")

Exportado a df_paises_vdem.xlsx
